In [1]:
import pandas as pd
import re
from datetime import datetime
import numpy as np
from plotnine import ggplot, geom_bar, aes, geom_col, labs, theme, element_blank, element_text

In [2]:
LATESTNCBI = "/Users/claramal/Desktop/Moncla_Lab/H5_avian_ecology/metadata/sequence_metadata/sequence_metadata_07012024.csv"

In [3]:
data_ncbi = pd.read_csv(LATESTNCBI, sep = ",")

In [4]:
# Function 1: making Collection Date a datetime object
def format_date(datestr):
    i = str(datestr)
    date_object = datetime.strptime(i, '%Y-%m-%d')
    return (date_object)

In [5]:
# Function 2: Extracting strain name from GenBank Title
def strain_name(df):
    df["strain"] = df["GenBank_Title"].str.extract(pat = r"(A[\/].*[\/]\d{4})", flags = 0, expand = True)
    df["strain"] = df["strain"].str.lower()
    df["strain"] = df["strain"].str.replace("-", "")
    df["strain"] = df["strain"].str.replace(" ", "")
    return

In [6]:
# Function 3: Detemining duplicate strains
def to_dedupe(df):
    strain_count = df["strain_segment"].value_counts()
    strain_count = strain_count.rename_axis("strain_segment").reset_index()
    duplicate_strains = []
    for strain, count in strain_count.iterrows():
        if count['count'] > 1:
            duplicate_strains.append(count['strain_segment'])
    return(print(duplicate_strains))

In [7]:
#Function 4: write a new FASTA with updated header (function written by Maria)  
def fasta_writer(path, filename, df):
            
    try:  
        os.mkdir(path)

    except OSError as error:
        pass

    with open(f"{path}{filename}", "w") as f:
        for index, row in df.iterrows():
            f.write(f"{row['header']}\n")
            f.write(f"{row['sequence']}\n")

In [8]:
# The following are a number of dictionaries necessary for mapping onto the metadata

State_Latitude_Dict = {"alabama":32.7784,
                       "alaska":64.0610,
                       "arizona":34.2740,
                       "arkansas":34.8938,
                       "california":37.1992,
                       "colorado":38.9979,
                       "connecticut":41.6220,
                       "delaware":38.9895,
                       "florida":28.6282,
                       "georgia":32.6405,
                       "hawaii":20.2951, 
                       "idaho":44.3538,
                       "illinois":40.0411,
                       "indiana":39.8938,
                       "iowa":42.0753,
                       "kansas":38.4949,
                       "kentucky":37.5360,
                       "louisiana":31.0689,
                       "maine":45.3702,
                       "maryland":39.0550,
                       "massachusetts":42.2596,
                       "michigan":44.3461,
                       "minnesota":46.2810,
                       "mississippi":32.7351,
                       "missouri":38.3568,
                       "montana":47.0527,
                       "nebraska":41.5392,
                       "nevada":39.3306,
                       "newhampshire":43.6805,
                       "newjersey":40.1907,
                       "newmexico":34.4066,
                       "newyork":42.9543,
                       "northcarolina":35.5579,
                       "northdakota":47.4499,
                       "ohio":40.2863,
                       "oklahoma":35.5910,
                       "oregon":43.9336,
                       "pennsylvania":40.8786,
                       "rhodeisland":41.6762,
                       "southcarolina":33.9169,
                       "southdakota":44.4446,
                       "tennessee":35.8605,
                       "texas":31.4789,
                       "utah":39.3057,
                       "vermont":44.0688,
                       "virginia":37.5229,
                       "washington":47.3822,
                       "westvirginia":38.6409,
                       "wisconsin":44.6243,
                       "wyoming":42.9959,
                       "temascalapa":19.8301, 
                       "sonora":29.3846, 
                       "libres":19.4663,
                       "guanajuato":21.0186,
                       "tehuitzingo":18.3324,
                       "texcoco":19.5135,
                       "veracruz":19.1974} 

State_Longitude_Dict = {"alabama":-86.8290,
                       "alaska":-152.2307,
                       "arizona":-111.6582,
                       "arkansas":-92.4429,
                       "california":-119.4505,
                       "colorado":-105.5495,
                       "connecticut":-72.7272,
                       "delaware":-75.5051,
                       "florida":-82.4426,
                       "georgia":-83.4418,
                       "hawaii":-156.3720, 
                       "idaho":-114.6086,
                       "illinois":-89.1965,
                       "indiana":-86.2818,
                       "iowa":-93.4959,
                       "kansas":-98.3801,
                       "kentucky":-85.3021,
                       "louisiana":-91.9952,
                       "maine":-69.2438,
                       "maryland":-76.7909,
                       "massachusetts":-71.8083,
                       "michigan":-85.4114,
                       "minnesota":-94.3046,
                       "mississippi":-89.6680,
                       "missouri":-92.4571,
                       "montana":-109.645,
                       "nebraska":-99.7968,
                       "nevada":-116.6268,
                       "newhampshire":-71.5818,
                       "newjersey":-74.6733,
                       "newmexico":-106.1128,
                       "newyork":-75.5262,
                       "northcarolina":-79.3856,
                       "northdakota":-100.4674,
                       "ohio":-82.7938,
                       "oklahoma":-97.4943,
                       "oregon":-120.5617,
                       "pennsylvania":-77.7985,
                       "rhodeisland":-71.5562,
                       "southcarolina":-80.8957,
                       "southdakota":-100.2282,
                       "tennessee":-86.3504,
                       "texas":-99.3287,
                       "utah":-111.6685,
                       "vermont":-72.6663,
                       "virginia":-78.8531,
                       "washington":-120.4505,
                       "westvirginia":-80.6230,
                       "wisconsin":-89.9941,
                       "wyoming":-107.5527,
                       "temascalapa":-98.9020, 
                       "sonora":-110.5208, 
                       "libres":-97.6835,
                       "guanajuato":-101.2596,
                       "tehuitzingo":-98.2750,
                       "texcoco":-98.8808,
                       "veracruz":-96.1385}

In [9]:
# Step 1: Clean and organize the metadata

In [10]:
print(len(data_ncbi))

13326


In [11]:
# Update the dates in data_ncbi

ncbi_filtered = data_ncbi[data_ncbi["Collection_Date"].str.match(r"^\d{4}-\d{2}-\d{2}$")]
#add day to data with just YYYY-MM

ncbi_filtered_1 = ncbi_filtered.copy()

ncbi_filtered_1["Collection_datetime"] = ncbi_filtered["Collection_Date"].apply(format_date)

print(type(ncbi_filtered_1["Collection_datetime"].dtype))

<class 'numpy.dtypes.DateTime64DType'>


In [13]:
print(len(ncbi_filtered_1)) #Length 12870 (removed 456 incorrect dates)

12870


In [14]:
# Make strain name in data_ncbi

ncbi_filtered_1.pipe(strain_name)

In [15]:
# Make a column that combines strain_segment

ncbi_filtered_1["Segment"] = ncbi_filtered_1["Segment"].astype(str)
# print(ncbi_filtered_1.Segment.value_counts())
ncbi_filtered_1["strain_segment"] = ncbi_filtered_1[["strain", "Segment"]].apply('_'.join,axis=1)

In [16]:
# Get state from strain name

# Extract state from strain name
strain_split = ncbi_filtered_1["strain"].str.split(pat = "/", expand = True)
#print(strain_split)

# Add State to ncbi_filtered_1
ncbi_filtered_1["State"] = strain_split[2]
#ncbi_filtered_1.head()

In [17]:
# Fix the State column

print(set(ncbi_filtered_1["State"]))

ncbi_stateadded = ncbi_filtered_1.copy()

ncbi_stateadded["State"] = ncbi_filtered_1["State"].replace(to_replace = ("az", "oh", "de", "md", "fl", "flordia", "louisana"), 
                                                                value = ("arizona", "ohio", "delaware", "maryland", "florida", "florida", "louisiana"))

print(set(ncbi_stateadded["State"]))

{'northdakota', 'newhampshire', 'nebraska', 'de', 'newjersey', 'maine', 'massachusetts', 'texcoco', 'alaska', 'connecticut', 'fl', 'veracruz', 'virginia', 'louisiana', 'libres', 'idaho', 'oh', 'utah', 'delaware', 'iowa', 'az', 'florida', 'texas', 'tennessee', 'newmexico', 'kentucky', 'newyork', 'northcarolina', 'arizona', 'alabama', 'oregon', 'temascalapa', 'southdakota', 'vermont', 'wisconsin', 'washington', 'maryland', 'colorado', 'kansas', 'oklahoma', 'wyoming', 'southcarolina', 'california', 'usa', 'minnesota', 'georgia', 'sonora', 'tehuitzingo', 'michigan', 'pennsylvania', 'indiana', 'louisana', 'montana', 'illinois', 'ohio', 'flordia', 'missouri', 'guanajuato', 'md'}
{'northdakota', 'newhampshire', 'nebraska', 'newjersey', 'maine', 'massachusetts', 'texcoco', 'alaska', 'connecticut', 'veracruz', 'virginia', 'louisiana', 'libres', 'idaho', 'utah', 'delaware', 'iowa', 'florida', 'texas', 'tennessee', 'newmexico', 'kentucky', 'newyork', 'northcarolina', 'arizona', 'alabama', 'oregon

In [18]:
# Just to check in at this point:
# I have filtered for just the sequences that have full YYYY-MM-DD dates: ncbi_filtered_1
# I have added a Collection_datetime column that is the same as Collection_date but in datetime format: ncbi_filtered_1
# I have added a column with strain title: ncbi_filtered_1
# I have added a column with strain_segment: ncbi_filtered_1
# I have added a column with corrected state data: ncbi_stateadded

In [19]:
# To make a list of bird orders are included in the metadata, first determine which bird species are in the metadata

ncbi_included_avian_hosts = ncbi_stateadded["Host"].value_counts()
ncbi_included_avian_hosts = ncbi_included_avian_hosts.rename_axis("Host").reset_index()
print(len(ncbi_included_avian_hosts)) # So far, length is 89 species/genera/etc., but always check to see if that number changes

#ncbi_included_avian_hosts.to_csv("/Users/claramal/Desktop/Moncla_Lab/H5_avian_ecology/metadata/ncbi_included_avian_hosts.csv", sep=",")

89


In [ ]:
# Optional: make a plot of the included avian hosts
ncbi_included_avian_hosts = ncbi_included_avian_hosts.sort_values(by = "count", ascending = False)
ncbi_included_avian_hosts.head()

# plot this using ggplot
(
    ggplot(ncbi_included_avian_hosts, aes(x = "Host", y = "count"))
    + geom_col()
    + labs(x = "Host Genus or Species", y = "Number of Sequences")
    + theme(axis_text_x = element_text(rotation = 90))
    + theme(figure_size=(11,5))
)

In [21]:
# Add a column for host bird order and domestic status

birdies = pd.read_csv("/Users/claramal/Desktop/Moncla_Lab/H5_avian_ecology/metadata/included_avian_hosts.csv", sep = ",")

ncbi_state_birdorder = pd.merge(ncbi_stateadded, birdies, on="Host")
print(len(ncbi_state_birdorder)) #Length of this is 12870

12870


In [22]:
# Add a column for flyways

# Upload flyways dataframe
flyways = pd.read_csv("/Users/claramal/Desktop/Moncla_Lab/H5_avian_ecology/metadata/helpful_lists/flywayregions.csv", sep = ",", dtype={'location':str})
column_mapper = {"location":"State"}
flyways = flyways.rename(mapper = column_mapper, axis = 1)
flyways["State"] = flyways["State"].str.lower()

# Add Mexican states to flyways dataframe
mexico_flyways = pd.DataFrame({"State":["temascalapa", "sonora", "libres", "guanajuato", "tehuitzingo", "texcoco", "veracruz"],
        "flyway":["Central Flyway", "Pacific Flyway", "Central Flyway", "Central Flyway", "Central Flyway", "Central Flyway", "Central Flyway"],
        "geo_region_na":["Mexico", "Mexico", "Mexico", "Mexico", "Mexico", "Mexico", "Mexico"]
       })
flyways_updt = flyways._append(mexico_flyways, ignore_index = True)

# Add flyways to metadata
ncbi_state_birdorder_flyways = pd.merge(ncbi_state_birdorder, flyways_updt, on = "State", how = "outer") #Use how = "outer" to keep state-less sequences
#ncbi_state_birdorder_flyways.head()
print(len(ncbi_state_birdorder_flyways)) #Length of this is ___

12899


In [24]:
# Add columns for latitide and longitude

ncbi_state_birdorder_flyways["State_Lat"] = ncbi_state_birdorder_flyways["State"].map(State_Latitude_Dict)
ncbi_state_birdorder_flyways["State_Long"] = ncbi_state_birdorder_flyways["State"].map(State_Longitude_Dict)
ncbi_state_birdorder_flyways["Centroid_Coord"] = ncbi_state_birdorder_flyways["State_Lat"].astype(str) + ',' + ncbi_state_birdorder_flyways["State_Long"].astype(str)

In [25]:
# Drop all the excess columns you made

drop_columns = ["count", "commonname", "broad", "notes", "geo_region_na", "Geo_Location", "State_Lat", "State_Long"]
ncbi_updated = ncbi_state_birdorder_flyways.drop(labels=drop_columns, axis = 1)
ncbi_updated.head()

,Accession,Organism_Name,SRA_Accession,Organization,Org_location,Release_Date,Isolate,Species,Molecule_type,Length,...,BioProject,GenBank_Title,Collection_datetime,strain,strain_segment,State,order,domesticstatus,flyway,Centroid_Coord
0,OQ957841.1,Influenza A virus,NaN,"United States Department of Agriculture, Agric...",USA,2023-08-23,22-004941-003,Alphainfluenzavirus influenzae,ssRNA(-),2280.0,...,NaN,Influenza A virus (A/American Wigeon/Alabama/2...,2022-02-05,a/americanwigeon/alabama/22004941003/2022,a/americanwigeon/alabama/22004941003/2022_1,alabama,anseriformes,wild,Mississippi Flyway,"32.7784,-86.829"
1,OQ957842.1,Influenza A virus,NaN,"United States Department of Agriculture, Agric...",USA,2023-08-23,22-004941-003,Alphainfluenzavirus influenzae,ssRNA(-),2274.0,...,NaN,Influenza A virus (A/American Wigeon/Alabama/2...,2022-02-05,a/americanwigeon/alabama/22004941003/2022,a/americanwigeon/alabama/22004941003/2022_2,alabama,anseriformes,wild,Mississippi Flyway,"32.7784,-86.829"
2,OQ957843.1,Influenza A virus,NaN,"United States Department of Agriculture, Agric...",USA,2023-08-23,22-004941-003,Alphainfluenzavirus influenzae,ssRNA(-),2151.0,...,NaN,Influenza A virus (A/American Wigeon/Alabama/2...,2022-02-05,a/americanwigeon/alabama/22004941003/2022,a/americanwigeon/alabama/22004941003/2022_3,alabama,anseriformes,wild,Mississippi Flyway,"32.7784,-86.829"
3,OQ957844.1,Influenza A virus,NaN,"United States Department of Agriculture, Agric...",USA,2023-08-23,22-004941-003,Alphainfluenzavirus influenzae,ssRNA(-),1704.0,...,NaN,Influenza A virus (A/American Wigeon/Alabama/2...,2022-02-05,a/americanwigeon/alabama/22004941003/2022,a/americanwigeon/alabama/22004941003/2022_4,alabama,anseriformes,wild,Mississippi Flyway,"32.7784,-86.829"
4,OQ957845.1,Influenza A virus,NaN,"United States Department of Agriculture, Agric...",USA,2023-08-23,22-004941-003,Alphainfluenzavirus influenzae,ssRNA(-),1497.0,...,NaN,Influenza A virus (A/American Wigeon/Alabama/2...,2022-02-05,a/americanwigeon/alabama/22004941003/2022,a/americanwigeon/alabama/22004941003/2022_5,alabama,anseriformes,wild,Mississippi Flyway,"32.7784,-86.829"


In [26]:
# Another check in:
# I have filtered for just the sequences that have full YYYY-MM-DD dates: ncbi_filtered_1
# I have added a Collection_datetime column that is the same as Collection_date but in datetime format: ncbi_filtered_1
# I have added a column with strain title: ncbi_filtered_1
# I have added a column with strain_segment: ncbi_filtered_1
# I have added a column with corrected state data: ncbi_stateadded
# I have added the corresponding host (bird) taxonomic order and domestic status: ncbi_state_birdorder
# I have addded the corresponding flyway data: ncbi_state_birdorder_flyways
# I have added the corresponding latitude and longitude: ncbi_updated

In [27]:
# Save as an updated TSV
ncbi_updated.to_csv("/Users/claramal/Desktop/Moncla_Lab/H5_avian_ecology/metadata/sequence_metadata/NCBIdata_07012024_updated.tsv", sep = '\t')

In [28]:
# Step 2: Deduplicate

In [31]:
# Check to see if each combination of strain and segment (there should be 8 segments for each strain) is unique

print(ncbi_updated["strain_segment"].is_unique)

False


In [32]:
# Which strains have to be deduplicated?

ncbi_updated.pipe(to_dedupe)

['a/redheadduck/northcarolina/w2483a/2024_8', 'a/canadagoose/louisiana/w23902/2023_1', 'a/redheadduck/northcarolina/w2483a/2024_6', 'a/redheadduck/northcarolina/w2483a/2024_7', 'a/redheadduck/northcarolina/w2483a/2024_5', 'a/canadagoose/louisiana/w23902/2023_7', 'a/canadagoose/louisiana/w23902/2023_3', 'a/canadagoose/louisiana/w23902/2023_2', 'a/redheadduck/northcarolina/w2483a/2024_4', 'a/canadagoose/louisiana/w23902/2023_5', 'a/canadagoose/louisiana/w23902/2023_4', 'a/canadagoose/louisiana/w23902/2023_6', 'a/canadagoose/louisiana/w23902/2023_8', "a/ross'sgoose/kansas/w23949a/2023_8", 'a/baldeagle/fl/w22114/2022_2', 'a/baldeagle/fl/w22114/2022_1', "a/ross'sgoose/kansas/w23949a/2023_1", "a/ross'sgoose/kansas/w23949a/2023_4", 'a/baldeagle/fl/w22114/2022_4', 'a/baldeagle/fl/w22114/2022_6', 'a/baldeagle/fl/w22114/2022_7', 'a/baldeagle/fl/w22114/2022_5', 'a/baldeagle/fl/w22114/2022_3', "a/ross'sgoose/kansas/w23949a/2023_2", 'a/baldeagle/fl/w22114/2022_8', "a/ross'sgoose/kansas/w23949a/2023

In [ ]:
# Reasons for choosing to delete duplicates
# a/canada goose/louisiana/w23902/2023 > arbitrary (same metadata) 
# a/redhead duck/north carolina/w2483a/2024 > arbitrary (same metadata)
# a/bald eagle/fl/w22114/2022 > oronasopharynx swab
# a/ross's goose/kansas/w23949a/2023 > non-passaged (not a clone)
# a/canada goose/north carolina/w2490a/2024 > arbitrary (same metadata)
# a/common eider/massachusetts/w22438d/2022 > arbitrary (same metadata)
# a/bluewinged teal/louisiana/ugai23893/2023 > arbitrary (same metadata)
# a/wood duck/south carolina/w24007e/2023 > arbitrary (same metadata)
# a/greater whitefronted goose/louisiana/w23857a/2023 > arbitrary (same metadata)
# a/wood duck/georgia/w24017/2023 > mostly arbitrary (same metadata), seg. 3 duplicate (longer)
# a/wood duck/north carolina/w24011a/2023 > longer sequence
# a/wood duck/georgia/w24007b/2023 > arbitrary (same metadata)

In [ ]:
# Remove duplicates by their indexes

ncbi_remove = [, #canadagoose/louisiana
, #redheadduck/northcarolina
, #baldeagle/florida
, #rossgoose/kansas 
, #canadagoose/northcarolina
, #commoneider/massachusetts
, #bluewingedteal/louisiana
, #woodduck/southcarolina
, #whitefrontedgoose/louisiana
, #woodduck/georgia_1of2
, #woodduck/northcarolina]
ncbi_updated_deduped = ncbi_updated.drop(label = ncbi_remove, axis = 0)

In [ ]:
# Save as TSV
ncbi_updated_deduped.to_csv(
    "/Users/claramal/Desktop/Moncla_Lab/H5_avian_ecology/metadata/sequence_metadata/NCBIdata_07012024_updated_deduped.tsv", sep = '\t')

In [ ]:
# Step 3: Make FASTA headers from this metadata

In [ ]:
#read FASTA as datframe (code courtesy of Maria)
FASTANAME = "NAMEHERE"

fasta_data = []
    
with open(FASTANAME) as f:
    header = ""
    sequence = ""
    for line in f:
        if line.startswith(">"):
            if header != "":
                fasta_data.append({"header": header, "sequence": sequence})
            header = line.strip() 
            sequence = ""
        else:
            sequence += line.strip()
    fasta_data.append({"header": header, "sequence": sequence}) #last line

df = pd.DataFrame(fasta_data)

print(df.head())

In [ ]:
#split header into two columns

fa_header_split = df["header"].str.split(pat = ' |', expand = True)
df["Accession"] = fa_header_split[0]
df["Accession"] = df["Accession"].str.replace(">", "")
print(df.head())
#get rid of second half, new column with just accession

In [ ]:
#merge metadata and fasta file on Accession 
merged = pd.concat([metadata, df], axis = 1)
merged.head()

In [ ]:
#new header column with strain
df_new["header"] = merged["strain"]
df_new["header"] = ">" + df_new["header"]
df_new.head()
#if human: df['header'] = df[['Strain', 'Accession', 'Subtype', 'Date', 'Host', 'country', 'region','ph1', 'ph2', 'ph3']].apply('|'.join, axis=1)

In [ ]:
# Write the new FASTA

fasta_writer("/Users/claramal/Desktop/Moncla_Lab/H5_avian_ecology/raw_data/", ".fasta", df)